# Workflow 1 – Operational Protein Units (OPUs)

**Associated manuscript:** Huber et al. – *Ocean Microbiome Signatures capturing the spatial organization of metabolic potential in the global sunlit ocean* –(under review)

**Description:** This notebook covers the full OPU analysis pipeline:
- Definition and annotation of OPUs (functional and taxonomic)
- Global distribution and biogeography (distance decay, ocean basin composition)
- Core vs. Endemic OPU characterization
- Functional diversity and richness across latitude

**Input files:** All input files correspond to Supplementary Tables (S1–S7). Place them in the same directory as this notebook.

**Output files:** Intermediate and final tables are saved as CSV/TSV in the working directory.

>  **Note:** Two steps require R (Mantel correlogram, UpSet plot). The R code is provided as markdown cells within this notebook.

### **1. Load Required Packages**


In [ ]:
import pandas as pd
import re
import numpy as np
import seaborn as sns

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib import rcParams

# Import cartopy for spatial mapping
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from geopy.distance import geodesic

from scipy.spatial.distance import pdist, squareform, braycurtis
from scipy.optimize import curve_fit
from scipy.stats import pearsonr, spearmanr, ks_2samp, entropy

from skbio.stats.composition import ancom
from skbio.diversity.alpha import chao1

# Additional libraries for advanced plots
from pycirclize import Circos
import scikit_posthocs as sp

# Set plotting defaults
rcParams['pdf.fonttype'] = 42  # Ensures text is saved as text, not paths
rcParams['font.family'] = 'Helvetica'

### **2. Load Input Tables**  

- **Metadata** (`metadata_df`) 

> **Input:** `TableS1_metadata.csv` – Metadata for 1,379 metagenomic samples (Table S1). Columns include sample ID, ocean basin, coordinates, expedition, and community-level diversity indices.

In [ ]:
metadata_df = pd.read_csv("TableS1_metadata.csv", delimiter=",")
num_filas = metadata_df.shape[0]
metadata_df

- **OPUs Abundance Table** (`OPUsAb_df`)  

> **Input:** `TableS2_OPUs_abundance.csv` – OPU abundance matrix (419,979 OPUs × 1,379 samples), normalized and rarefied to 2,100 counts per sample (Table S2).

In [ ]:
OPUsAb_df = pd.read_csv("TableS2_OPUs_abundance.csv", delimiter=",")
num_filas = OPUsAb_df.shape[0]
OPUsAb_df

- **OPUs Features and Functional Annotation Table** (`OPUsFea_df`)  

> **Input:** `TableS3_OPUs_annotation.csv` – Functional (KEGG KO) and taxonomic (GTDB via CAT) annotation of all 419,979 OPUs (Table S3).

In [ ]:
OPUsFea_df = pd.read_csv("TableS3_OPUs_annotation.csv", delimiter=",")
num_filas = OPUsFea_df.shape[0]
OPUsFea_df

- **MAGs Table** (`mg`)  

> **Input:** `TableS7_MAGs.txt` – 20,482 marine MAGs with genome size and quality metrics (Table S7). Used in Section 4.3.2.

In [ ]:
mg = pd.read_csv("TableS7_MAGs.txt", delimiter= "\t")
mg

- **Key KO Reference Table** (`keyKO`)  

> **Input:** `TableS5_KeyKOdb.csv` – KeyKOdb: curated reference database of marker KEGG Orthology entries for core metabolic pathways (Table S5).

In [ ]:
keyKO = pd.read_csv("TableS5_KeyKOdb.csv")
keyKO

### **3. Spatial Distribution of Metagenomes**

  - **3.1** Compute the number of samples per oceanographic expedition.  

> Counts the number of samples contributed by each oceanographic expedition (stored in the `Note` column of the metadata). Used to report the composition of the AtlantECO-BASEv1 dataset (Supplementary Fig. S1).

In [ ]:
campaign_counts = metadata_df['Note'].value_counts()
print(campaign_counts)

  - **3.2** Generate a distribution map of metagenomic samples.  

> Generates a global map of the 1,379 metagenomic sampling stations using Cartopy. Corresponds to **Supplementary Fig. S1**.

In [ ]:
projection = ccrs.PlateCarree()

fig = plt.figure(figsize=(12, 6))
ax = fig.add_subplot(1, 1, 1, projection=projection)

ax.add_feature(cfeature.LAND)
ax.add_feature(cfeature.COASTLINE)
#ax.add_feature(cfeature.BORDERS, linestyle=':')
#ax.add_feature(cfeature.LAKES, alpha=0.5)
ax.add_feature(cfeature.RIVERS)

ax.set_extent([-180, 180, -90, 90], crs=ccrs.PlateCarree())

gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 10, 'color': 'gray'}
gl.ylabel_style = {'size': 10, 'color': 'gray'}

campaign_colors = {
    'GO-SHIP DNA samples  prokaryotes rich': '#3498db',
    "OSD 2014  DNA samples corresponding to size fractions for prokaryotes": "#8e44ad",
    "Marine metagenomes from the bioGEOTRACES project prokaryotes rich": "#e74c3c",          
    "Tara Oceans Polar Cicle DNA samples corresponding to size fractions for prokaryotes": "#16a085",
    "Amazon Continuum Metagenomes samples prokaryotes rich": "#2ecc71",
    'Tara Oceans DNA samples corresponding to size fractions for prokaryotes': '#1a5276',
}

custom_labels = {
    "GO-SHIP DNA samples  prokaryotes rich": 'GO-SHIP [N=928]',
    "OSD 2014  DNA samples corresponding to size fractions for prokaryotes" : "OSD-2014 [N=126]",
    "Marine metagenomes from the bioGEOTRACES project prokaryotes rich" : "bioGEOTRACES [N=62]",            
    "Tara Oceans Polar Cicle DNA samples corresponding to size fractions for prokaryotes" : "Tara Polar [N=28]",
    "Amazon Continuum Metagenomes samples prokaryotes rich" : "Anaconda [N=19]",
    'Tara Oceans DNA samples corresponding to size fractions for prokaryotes': 'Tara Ocean [N=94]'
}

for campaign in campaign_colors.keys():
    campaign_data = metadata_df[metadata_df['Note'] == campaign]
    ax.scatter(campaign_data['decimalLongitude'], campaign_data['decimalLatitude'], 
               label=custom_labels[campaign], color=campaign_colors[campaign], edgecolor='#1c2833', linewidths=0.7, 
               alpha=0.7, s=100, transform=ccrs.PlateCarree())

ax.legend(title='Oceanic Campaign', loc='lower right')

plt.title('Samples Location')

plt.savefig('samples_distribution_map.png', dpi=300, bbox_inches='tight')

plt.show()

### **4. OPUs Functional Characterization & Global Distribution**  

#### **4.1 OPUs Functional Annotation: Known vs. Dark Matter**

- **4.1.1** Quantify the **number, abundance, and relative contribution** of known vs. unknown OPUs.  

> Quantifies the number and total abundance of **Known** (KO-annotated) vs. **Unknown** (dark matter) OPUs. Results are reported in the main text (Section: *Over 400,000 OPUs delineate a small functional core*).

In [ ]:
#Number and abundance
OPUs_c = OPUsFea_df.groupby('OPU_Type').agg({
    'OPU': 'count',                # Count the number of OPUs
    'Total_Abundance': 'sum'       # Sum the total abundance
}).reset_index()
# Rename columns for clarity
OPUs_c.columns = ['OPU_Type', 'Number_of_OPUs', 'Total_Abundance']
# Display the result
OPUs_c

In [ ]:
# Compute relative contribution of Known vs. Unknown OPUs
# These values are derived from the groupby aggregation above (OPUs_c)
n_unknown = OPUs_c.loc['Unknown', 'OPU']
n_known   = OPUs_c.loc['Known',   'OPU']
ab_unknown = OPUs_c.loc['Unknown', 'Total_Abundance']
ab_known   = OPUs_c.loc['Known',   'Total_Abundance']

total      = n_unknown + n_known
totalCont  = ab_unknown + ab_known
pct_unknown_count = (n_unknown / total) * 100
pct_unknown_abund = (ab_unknown / totalCont) * 100

print(f'Unknown OPUs: {n_unknown:,} ({pct_unknown_count:.1f}% of total)')
print(f'Unknown abundance contribution: {pct_unknown_abund:.1f}%')

- **4.1.2** Explore the **prevalence and abundance distributions** of known vs. unknown OPUs.  

In [ ]:
# Split table in Known and Unknown OPUs 
unknown = OPUsFea_df[OPUsFea_df['unknown_percentage'] == 100]
known = OPUsFea_df[OPUsFea_df['unknown_percentage'] < 100]

In [ ]:
# Flatten the dataframes to create two 1D arrays of abundance values
unknown_abundance = unknown ["Total_Abundance"]
known_abundance = known ["Total_Abundance"]
unknown_abundance_flat = unknown_abundance.values.flatten()
known_abundance_flat = known_abundance.values.flatten()

In [ ]:
#Flatten the dataframes to create two 1D arrays of prevalence values
unknown_prevalence = unknown ["Occurrence"]
known_prevalence = known ["Occurrence"]
unknown_prevalence_flat = unknown_prevalence.values.flatten()
known_prevalence_flat = known_prevalence.values.flatten()

> Runs a **Kolmogorov-Smirnov (KS) test** to compare the abundance and prevalence distributions of Known vs. Unknown OPUs. The resulting plot corresponds to **Supplementary Fig. S2**.

In [ ]:
# Run KS test for abundance and prevalence ( Kolmogorov-Smirnov test (KS test) in Python. The KS test evaluates whether two samples come from the same distribution)
ks_stat_abundance, p_value_abundance = ks_2samp(unknown_abundance_flat, known_abundance_flat)
ks_stat_prevalence, p_value_prevalence = ks_2samp(unknown_prevalence_flat, known_prevalence_flat)

#Print results
print(f"KS Test for Abundance: Statistic = {ks_stat_abundance}, p-value = {p_value_abundance}")
print(f"KS Test for Prevalence: Statistic = {ks_stat_prevalence}, p-value = {p_value_prevalence}")

# Interpretation
if p_value_abundance < 0.05:
    print("Significant difference in abundance distributions (p < 0.05).")
else:
    print("No significant difference in abundance distributions (p >= 0.05).")

if p_value_prevalence < 0.05:
    print("Significant difference in prevalence distributions (p < 0.05).")
else:
    print("No significant difference in prevalence distributions (p >= 0.05).")

In [ ]:
# Plot the distributions
rcParams['pdf.fonttype'] = 42  # Ensures text is saved as text, not paths
rcParams['font.family'] = 'Helvetica'

fig, axs = plt.subplots(1, 2, figsize=(12, 5))

# Plot Prevalence Distribution
axs[0].hist(unknown_prevalence_flat, bins=20, alpha=0.5, range=(0, 70),label='Unknown OPUs', color='blue')
axs[0].hist(known_prevalence, bins=20, alpha=0.5, range=(0, 70), label='Known OPUs', color='green')
axs[0].set_title('Prevalence Distribution')
axs[0].set_xlabel('Prevalence (%)')
axs[0].set_ylabel('Frequency')
axs[0].set_yticks([0, 20000, 40000])
axs[0].legend()

# Plot Abundance Distribution
axs[1].hist(unknown_abundance, bins=20, alpha=0.5, range=(0, 100), label='Unknown OPUs', color='blue')
axs[1].hist(known_abundance, bins=20, alpha=0.5, range=(0, 100), label='Known OPUs', color='green')
axs[1].set_title('Abundance Distribution')
axs[1].set_xlabel('Abundance')
axs[1].set_ylabel('Frequency')
axs[1].set_yticks([0, 50000, 150000])
axs[1].legend()

# Display KS test results in the title of each plot
axs[0].set_title(f'Prevalence Distribution (KS p-value: {p_value_prevalence:.3e})')
axs[1].set_title(f'Abundance Distribution (KS p-value: {p_value_abundance:.3e})')

plt.tight_layout()

# Print KS test results
print(f"KS test for Prevalence: p-value = {p_value_prevalence:.3e}")
print(f"KS test for Abundance: p-value = {p_value_abundance:.3e}")

# Plot the distributions
rcParams['pdf.fonttype'] = 42  # Ensures text is saved as text, not paths
rcParams['font.family'] = 'Helvetica'

fig, axs = plt.subplots(1, 2, figsize=(12, 5))

# Plot Prevalence Distribution
axs[0].hist(unknown_prevalence_flat, bins=20, alpha=0.5, range=(0, 70),label='Unknown OPUs', color='blue')
axs[0].hist(known_prevalence, bins=20, alpha=0.5, range=(0, 70), label='Known OPUs', color='green')
axs[0].set_title('Prevalence Distribution')
axs[0].set_xlabel('Prevalence (%)')
axs[0].set_ylabel('Frequency')
axs[0].set_yticks([0, 20000, 40000, 60000, 120000])
axs[0].legend()

# Plot Abundance Distribution
axs[1].hist(unknown_abundance, bins=20, alpha=0.5, range=(0, 100), label='Unknown OPUs', color='blue')
axs[1].hist(known_abundance, bins=20, alpha=0.5, range=(0, 100), label='Known OPUs', color='green')
axs[1].set_title('Abundance Distribution')
axs[1].set_xlabel('Abundance')
axs[1].set_ylabel('Frequency')
axs[1].set_yticks([0, 50000, 150000, 200000, 250000])
axs[1].legend()

# Display KS test results in the title of each plot
axs[0].set_title(f'Prevalence Distribution (KS p-value: {p_value_prevalence:.3e})')
axs[1].set_title(f'Abundance Distribution (KS p-value: {p_value_abundance:.3e})')

plt.tight_layout()
# Print KS test results
print(f"KS test for Prevalence: p-value = {p_value_prevalence:.3e}")
print(f"KS test for Abundance: p-value = {p_value_abundance:.3e}")

# Save the plot as a PDF (before showing the plot)
#output_pdf = "prevalence_abundance_distribution_UNK_KNW.pdf"  # Define the PDF file name
plt.savefig(output_pdf, format='pdf', dpi=300)
# Display the plot
plt.show()

#### **4.2 OPUs Global Distribution**

- **4.2.1** Perform **Distance Decay Analysis** and compute a **Mantel correlogram**.  

- Compute Bray-Curtis dissimilarity matrix

> **⚠️ Computationally intensive.** This cell computes the full pairwise Bray-Curtis dissimilarity matrix for 1,379 samples. It may take several hours depending on available RAM.

> **Output:** `bray_curtis_1379_samples.csv` – used in the distance decay analysis and Mantel correlogram below.

In [ ]:
## [NOTE]This analysis is highly computationally demanding. 

#  Remove the 'OPU' column (the first column) to keep only the abundance data
df_abundance = OPUsAb_df.drop(columns=['OPU'])

# Compute Bray-Curtis dissimilarity matrix
bray_curtis_matrix = pdist(df_abundance.T, metric='braycurtis')  # Transpose the dataframe to have samples as rows

# Convert the pairwise distances into a square matrix form
bray_curtis_square = squareform(bray_curtis_matrix)

# Convert back to a DataFrame for easier visualization
bray_curtis_df_tr = pd.DataFrame(bray_curtis_square_tr, index=df_abundance.columns, columns=df_abundance.columns)

# Display the resulting Bray-Curtis dissimilarity matrix
bray_curtis_df
#bray_curtis_df_tr.to_csv('bray_curtis_1379_samples.csv', index=False)

In [ ]:
#bray_curtis_df = pd.read_csv("bray_curtis_1379_samples.csv", delimiter=",", index_col=0)
bray_curtis_array = bray_curtis_df.values

# Extract the upper triangular part of the matrix (excluding the diagonal)
braycurtis_flat = bray_curtis_array[np.triu_indices_from(bray_curtis_array, k=1)]

# Calculate the average Bray-Curtis dissimilarity
average_dissimilarity = np.mean(braycurtis_flat)

# Print the result
print(f"The average Bray-Curtis dissimilarity is: {average_dissimilarity}")

- Distance Decay Analysis Mantel correlogram

> Loads the precomputed Bray-Curtis matrix and computes the **geographic distance matrix** (Euclidean, from decimal lat/lon). Fits an exponential decay model to the distance-decay relationship. Corresponds to **Fig. 2a** and main text description of spatial turnover.

In [ ]:
# Calculate the geographic distance matrix using Euclidean distance
coords = metadata_df[['decimalLatitude', 'decimalLongitude']].values
dist_matrix = np.zeros((len(metadata_df), len(metadata_df)))

for i in range(len(metadata_df)):
    for j in range(i + 1, len(metadata_df)):
        dist = geodesic((metadata_df.iloc[i]['decimalLatitude'], metadata_df.iloc[i]['decimalLongitude']),
                        (metadata_df.iloc[j]['decimalLatitude'], metadata_df.iloc[j]['decimalLongitude'])).kilometers
        dist_matrix[i, j] = dist
        dist_matrix[j, i] = dist
#dist_matrix.to_csv('distance_matrix_1379_tr_samples.csv', index=True)


In [ ]:
dist_matrix = pd.read_csv('distance_matrix_1379_tr_samples.csv', index_col= 0)
dist_matrix = np.array(dist_matrix)
geo_distance_flat = dist_matrix[np.triu_indices_from(dist_matrix, k=1)]

In [ ]:
# Define an exponential decay function
def exp_decay(x, a, b):
    return a * np.exp(-b * x)

# Fit the exponential decay model to the data
params, _ = curve_fit(exp_decay, geo_distance_flat, braycurtis_flat)

# Plot the fitted curve with the scatter plot
x_vals = np.linspace(geo_distance_flat.min(), geo_distance_flat.max(), 100)
y_vals = exp_decay(x_vals, *params)

plt.figure(figsize=(8, 6))
plt.scatter(geo_distance_flat, braycurtis_flat, alpha=0.7, label='Data')
plt.plot(x_vals, y_vals, color='red', label='Fitted Exponential Decay')
plt.xlabel('Geographic Distance (km)')
plt.ylabel('Bray-Curtis Dissimilarity')
plt.title('Distance Decay with Fitted Exponential Model')
plt.legend()
plt.savefig('distance_decay.png', dpi=300, bbox_inches='tight')

plt.show()

> **R code (run externally):** The Mantel correlogram was computed in R using the `vegan` package. Significant spatial autocorrelation was detected at short geographic ranges (p < 0.05). Results are reported in the main text.

```r
# Mantel Correlogram — run in R
# This code was executed externally in R. Results are described in the main text.
#
library(vegan)
library(picante)

bc  <- read.table('bray_curtis_1379_samples.csv', sep=',', header=TRUE, row.names=1)
geo <- read.table('distance_matrix_1379_tr_samples.csv', sep=',', header=TRUE, row.names=1)

mantel_corr <- mantel.correlog(bc, geo, nperm=999)
plot(mantel_corr)
```

- **4.2.2** Assess OPU composition at the **ocean basin level**:  


> **Subsampling rationale:** Ocean basins differ substantially in the number of available samples. To avoid biasing comparisons, basins with more samples are randomly subsampled to a maximum of 209 samples (the average per basin). This ensures equal statistical power across basins.

  - **Subsampling:** Since sample sizes vary across ocean basins, we subsample to a maximum of **172 samples per basin** (the dataset-wide average).  

In [ ]:
# Count the number of samples from each ocean
ocean_counts = metadata_df['Ocean'].value_counts()
# Print the counts
print(ocean_counts)

In [ ]:
# Filter and randomly subsample the ocean samples with more than 172 samples
num_samples = int(metadata_df["Ocean"].value_counts().mean())  # average samples per basin (~209)

Indian_samples = metadata_df[metadata_df['Ocean'] == 'Indian'].sample(n=num_samples, random_state=42)
North_Atlantic_samples = metadata_df[metadata_df['Ocean'] == 'North Atlantic'].sample(n=num_samples, random_state=42)
South_Atlantic_samples =metadata_df[metadata_df['Ocean'] == 'South Atlantic']
South_Pacific_samples = metadata_df[metadata_df['Ocean'] == 'South Pacific']
North_Pacific_samples = metadata_df[metadata_df['Ocean'] == 'North Pacific']
Antarctic_samples = metadata_df[metadata_df['Ocean'] == 'Antarctic']
Mediterranean_samples = metadata_df[metadata_df['Ocean'] == 'Mediterranean']
Arctic_samples = metadata_df[metadata_df['Ocean'] == 'Arctic']

# Concatenate the subsampled samples into a new DataFrame.
subsampled_df = pd.concat([Indian_samples, North_Atlantic_samples, South_Atlantic_samples, South_Pacific_samples, North_Pacific_samples, Antarctic_samples, Mediterranean_samples, Arctic_samples])
subsampled_df.reset_index(drop=True, inplace=True)
print(subsampled_df.shape)

In [ ]:
#Check!
# Count the number of samples from each ocean
ocean_counts = subsampled_df['Ocean'].value_counts()
print(ocean_counts)

In [ ]:
### Create a subsample abundance table
# Extract the list of MGnifyAssemblyNumb from subsampled_df.
sample_ids = subsampled_df['MGnifyAssemblyNumb'].tolist()

> **Output:** `abundance_subsample_df` – subsampled OPU abundance table used in all downstream basin-level comparisons.

In [ ]:
# Filter the abundance table using the sample IDs.
OPUsAb_df_in = OPUsAb_df.set_index(OPUsAb_df.columns[0])
abundance_subsample_df = OPUsAb_df_in.T.reindex(sample_ids)
abundance_subsample_df

In [ ]:
# Compute average OPU abundance per Ocean
average_per_ocean = merged.groupby('Ocean').mean()
# The result will have OPUs as rows and oceans as columns
average_per_ocean

In [ ]:
# Transpose: rows = OPUs, columns = Ocean basins
average_per_oceanT = average_per_ocean.T
average_per_oceanT.to_csv('OPUs_average_abundance_per_ocean.csv')

> **Output:** `OPUs_average_abundance_per_ocean.csv` – average OPU abundance per ocean basin.  
> This table is used as input for the **Kruskal-Wallis test + Dunn's post-hoc test** in Section 4.2.3 (Table S4).

- **4.2.3 Unique vs. Core OPUs Functional Characterization**  

  - A) Process data and generate plots for **Unique and Core OPUs**.  

In [ ]:
# From subsampled table create a binary table An OPU is considered present (1) if its abundance > 0 in a given sample.
binary_table = abundance_subsample_df.apply(lambda x: (x > 0).astype(int))
binary_table

In [ ]:
#in case you want to check
abundance_sum = abundance_subsample_df["OPU_1"].sum()
abundance_sum

In [ ]:
# Calculate the number of OPUs per ocean and generate a new DataFrame for every ocean indicating the OPU presence or absence  
# Create a list containing sample ID and ocean information
sample_ocean_list = subsampled_df[['MGnifyAssemblyNumb', 'Ocean']]

# Ensure sample IDs from metadata are present as columns in the OTU table
subAbun_T = abundance_subsample_df.T
subAbun_T
sample_ids = sample_ocean_list['MGnifyAssemblyNumb'].tolist()
sample_ids = [col for col in sample_ids if col in subAbun_T.columns]

# Iterate over ocean names and filter the OTU table based on the sample IDs for each ocean
ocean_tables = {}

for ocean in sample_ocean_list['Ocean'].unique():
    # Get sample IDs for the current ocean
    ocean_samples = sample_ocean_list.loc[sample_ocean_list['Ocean'] == ocean, 'MGnifyAssemblyNumb']
    ocean_samples = [sample for sample in ocean_samples if sample in sample_ids]
    # Filter the OTU table columns based on sample IDs
    ocean_otu = subAbun_T[ocean_samples]
    # Reset the index to move OTU names from index to a regular column
    ocean_otu_reset = ocean_otu.reset_index()
    # Rename the index column to 'OTU'
    ocean_otu_reset.rename(columns={'index':'OPU'}, inplace=True)
    # Check if any sample IDs were not present in the OTU table and print a warning
    missing_samples = set(ocean_samples) - set(ocean_otu.columns)
    if missing_samples:
        print(f"Warning: Sample IDs {missing_samples} are not present in the OTU table for ocean {ocean}.")
    # Store the filtered OTU table for the current ocean
    ocean_tables[ocean] = ocean_otu_reset


> Computes per-ocean statistics for each OPU: total abundance, average abundance, and occurrence count. These tables are merged into a single summary used to classify OPUs as Core or Endemic.

In [ ]:
#1.- Calculate the total abundance of each OTU across all samples in each ocean.
abundance_tables = {}
for ocean, ocean_otu_table in ocean_tables.items():
    # Calculate the sum of abundance for each OTU
    abundance_table = ocean_otu_table.copy()  # Make a copy to retain the 'OPU' column
    abundance_table['Abundance'] = abundance_table.drop(columns='OPU').sum(axis=1)
    abundance_table = abundance_table[['OPU', 'Abundance']]  # Reorder columns
    abundance_tables[ocean] = abundance_table

#2.- Count the number of occurrences of each OTU in each ocean.
occurrence_tables = {}
for ocean, ocean_otu_table in ocean_tables.items():
    # Count the occurrence of each OTU
    occurrence_table = ocean_otu_table.copy()  # Make a copy to retain the 'OPU' column
    occurrence_table['Occurrence'] = (occurrence_table.drop(columns='OPU') > 0).sum(axis=1)
    occurrence_table = occurrence_table[['OPU', 'Occurrence']]  # Reorder columns
    occurrence_tables[ocean] = occurrence_table

#3.- Merge the calculated abundance and occurrence data into a single table."
AVGabundance_tables = {}
# Loop over ocean tables
for ocean, ocean_otu_table in ocean_tables.items():
    # Calculate the average abundance for each OPU
    avg_abundance_table = ocean_otu_table.copy()  # Make a copy to retain the 'OPU' column
    avg_abundance_table['AVGabundance'] = avg_abundance_table.drop(columns='OPU').mean(axis=1)

    # Reorder columns to retain 'OPU' first, then 'AVGabundance'
    avg_abundance_table = avg_abundance_table[['OPU', 'AVGabundance']]

    # Store the result in the dictionary with the ocean name as the key
    AVGabundance_tables[ocean] = avg_abundance_table

#4.- Join the information in singles df
joined_tableA = pd.DataFrame()
for ocean, abundance_table in abundance_tables.items():
    # Join abundance and occurrence information for each ocean
    ocean_info = pd.merge(abundance_table, abundance_tables[ocean], on='OPU', how='inner')
    ocean_info['Ocean'] = ocean  # Add Ocean column
    joined_tableA = pd.concat([joined_tableA, ocean_info])

joined_tableO = pd.DataFrame()
for ocean, ocurrence_table in occurrence_tables.items():
    # Join abundance and occurrence information for each ocean
    ocean_info = pd.merge(ocurrence_table, occurrence_tables[ocean], on='OPU', how='inner')
    ocean_info['Ocean'] = ocean  # Add Ocean column
    joined_tableO = pd.concat([joined_tableO, ocean_info])

joined_tableAV = pd.DataFrame()
for ocean, avg_table in AVGabundance_tables.items():
    # Join abundance and occurrence information for each ocean
    ocean_info = pd.merge(avg_table, AVGabundance_tables[ocean], on='OPU', how='inner')
    ocean_info['Ocean'] = ocean  # Add Ocean column
    joined_tableAV = pd.concat([joined_tableAV, ocean_info])

print(joined_tableAV.head(5))

In [ ]:
# If you want to check, you can count the occurrences of "OPU_100" in the "OPU" column
OPU_100_count = joined_tableO['OPU'].value_counts().get('OPU_100', 0)
print("Number of occurrences of 'OPU_100':", OPU_100_count)

In [ ]:
# Pivot for abundance
abundance_shortA = joined_tableA.pivot(index='OPU', columns='Ocean', values='Abundance_x')
# Pivot for occurrence
occurrence_shortO = joined_tableO.pivot(index='OPU', columns='Ocean', values='Occurrence_x')
# Pivot for avg_abundance
occurrence_shortAV = joined_tableAV.pivot(index='OPU', columns='Ocean', values='AVGabundance_x')

In [ ]:
#create a new dataframe with the avg abundance of each OPU in each ocean
occurrence_shortAV = joined_tableAV.pivot(index='OPU', columns='Ocean', values='AVGabundance_x')
occurrence_shortAV

In [ ]:
#Using df_occurrence DataFrame containing the occurrence information for each OPU in each ocean
#Create a binary table
binary_df = occurrence_shortO.applymap(lambda x: 1 if x > 0 else 0)
binary_df
#binary_df.to_csv("OPUS_occurrence_perocean_binary_table_subsetfrom1379.csv")

> Loads the precomputed binary occurrence table. **Core OPUs** are present in all ocean basins; **Endemic OPUs** are restricted to a single basin. Results are reported in **Fig. 2b–c** and main text.

In [ ]:
binary_df= pd.read_csv("OPUS_occurrence_perocean_binary_table_subsetfrom1379.csv", delimiter=",")
num_filas = binary_df.shape[0]
binary_df = binary_df.set_index("OPU")
binary_df 

In [ ]:
# Total OPUs #
totalOPUs = binary_df.sum()
totalOPUs

In [ ]:
#Unique OPUs
OPUS_UNIQUE = binary_df[binary_df.sum(axis=1)==1]
OPUS_UNIQUE = OPUS_UNIQUE.sum(axis=0)
OPUS_UNIQUE

> Computes the number of OPUs shared by exactly 2, 3, … 8 basins. Results used in **Fig. 2b** (stacked bar plot of OPU sharing).

In [ ]:
# Calculated and plot the number of Shared OPUs
shared_counts = binary_df.sum(axis=1)

# Count the number of OPUs shared by exactly 2, 3, ..., 8 basins
shared_summary = shared_counts.value_counts().sort_index()

# Filter for counts between 2 and 8 basins
shared_summary = shared_summary[shared_summary.index.isin(range(1, 9))]

# Display the summary
print(shared_summary)

shared_summary= {
    1: 155485,
    2: 113861,
    3: 57764,
    4: 29295,
    5: 14621,
    6: 3405,
    7: 706,
    8: 146
}

# Create a bar plot
# Set the font to a standard, recognizable font like Arial
rcParams['pdf.fonttype'] = 42  # Ensures text is saved as text, not paths
rcParams['font.family'] = 'Helvetica'

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(shared_summary.keys(), shared_summary.values(), color='lightgrey', edgecolor='black', linewidth=1.5)

# Label the plot
ax.set_xlabel("Number of Ocean Basins")
ax.set_ylabel("Number of OPUs")
#ax.set_title("Distribution of OPUs Shared Across Ocean Basins")

# Show value labels above each bar for clarity
for x, y in shared_summary.items():
    ax.text(x, y + 500, str(y), ha='center', va='bottom')

# Save the plot as a PDF (before showing the plot)
output_pdf = "Distribution of OPUs Shared Across Ocean Basins.pdf"  # Define the PDF file name
plt.savefig(output_pdf, format='pdf', dpi=300)

# Display the plot
plt.show()
# Optionally, save the values to a file if you want to review them later
#hist_values.to_csv("OPUS_Shared_Basins.csv", index=False)


> Builds a pairwise comparison matrix of shared OPUs between ocean basins. Used to generate the **Circos plot (Fig. 2c)**.

In [ ]:
#Compute the comparation table with number of OPUs shared between ocean basins
comparison_table = binary_df.T @ binary_df
i, j = np.triu_indices(comparison_table.shape[0], k=1)
comparison_table.values[i, j] = 0
comparison_table

In [ ]:
#Plot_the_table
from pycirclize import Circos

# Set the font to a standard, recognizable font like Arial
rcParams['pdf.fonttype'] = 42  # Ensures text is saved as text, not paths
rcParams['font.family'] = 'Helvetica'
circos = Circos.initialize_from_matrix(
    comparison_table,
    space=3,
    r_lim=(93, 100),
    cmap="tab10",
    ticks_interval=100500,
    label_kws=dict(r=94, size=12, color="white"),
)

#print(combination_matrix)
fig = circos.plotfig()
#fig.savefig("circos_plot_subset1379.pdf")

In [ ]:
UpSet Plot — run in R
# Run externally in R. Input: OPUS_occurrence_perocean_binary_table_subsetfrom1379.csv
library(UpSetR)
binary_df <- read.csv('OPUS_occurrence_perocean_binary_table_subsetfrom1379.csv',
                      row.names=1, check.names=FALSE)
binary_df[binary_df > 0] <- 1
upset(binary_df, nsets=8, order.by='freq'

  - B) Analyze *functional repertoire* variation across oceans:  

In [2]:
#Process data
binary_df = pd.read_csv("OPUS_occurrence_perocean_binary_table_subsetfrom1379.csv", index_col=0)
#Add to binary table the Category: Core or Unique (and the ocean)
core_mask = binary_df.sum(axis=1) == binary_df.shape[1]
core_mask
# Find OPUs unique to a single basin
unique_mask = binary_df.sum(axis=1) == 1
unique_mask
# Create a new DataFrame with the filtered OPUs
UnCore_df= binary_df[core_mask | unique_mask].copy()
# Function to assign "core" or the unique basin name
def assign_category(row):
    if row.sum() == binary_df.shape[1]:  # Core OPUs (present in all basins)
        return "core"
    elif row.sum() == 1:  # Unique OPUs (present in only one basin)
        return binary_df.columns[row.argmax()]
    else:
        return None  # Should not happen

# Apply the function to create the "Category" column
UnCore_df["Category"] = UnCore_df.apply(assign_category, axis=1)

# Save or inspect the filtered data
UnCore_df
#UnCore_df.to_csv("Unique_Core_OPUs_abund_n.csv")

In [ ]:
# Merge the tables based on the OPU index, ensuring all OPUs from annotation_df are retained
OPUsFea_df_in = OPUsFea_df.set_index(OPUsFea_df.columns[0])
UnCoreFunc_df = OPUsFea_df_in.merge(UnCore_df[['Category']], left_index=True, right_index=True, how="left")

# Fill missing categories with 'NA' without using inplace
UnCoreFunc_df['Category'] = UnCoreFunc_df['Category'].fillna('NA')
UnCoreFunc_df = UnCoreFunc_df[UnCoreFunc_df['Category'] != 'NA']
UnCoreFunc_df
# Save the merged table
#UnCoreFunc_df.to_csv("Unique_Core_OPUs_functionalAssignation_155631X9_n.csv")

- ***Functional annotation Using  keyKO***

In [ ]:
keyKO = pd.read_csv("TableS5_KeyKOdb.csv")
keyKO

In [ ]:
UnCoreFunc_df = pd.read_csv("Unique_Core_OPUs_functionalAssignation_155631X9_n.csv")
UnCoreFunc_df = UnCoreFunc_df.set_index("OPU")
UnCoreFunc_df

In [ ]:
Unq_Cos = UnCoreFunc_df.join(
    keyKO.set_index("KO_id"), 
    on="Best_KO"
)
Unq_Cos

In [ ]:
# Replace NaN values with 'Unknown' where OPU_Type is 'Unknown'
Unq_Cos.loc[Unq_Cos['OPU_Type'] == 'Unknown', :] = Unq_Cos.loc[Unq_Cos['OPU_Type'] == 'Unknown', :].fillna('Unknown')
# Create a new dataframe by dropping rows where 'Functional hierarchy 1' is NaN
Un_Cos_KeyKOUnk= Unq_Cos.dropna(subset=['Functional hierarchy 1'])

- **Unique OPUs:**  
      - Assess functional diversity variation.  

In [ ]:
#Un_Cos_KeyKOUnk = pd.read_csv("Unique_Core_OPUs_KeyKO_Unkn_annotation_71436X15_n.csv")
# Filter the rows where 'Category' is not 'core'
unique_table = Un_Cos_KeyKOUnk[Un_Cos_KeyKOUnk['Category'] != 'core']
unique_table.shape
# Select only the Functional hierarchy 1 column
unique_table = unique_table[['Functional hierarchy 1']]
OPUsAb_df_in = OPUsAb_df.set_index("OPU")
# Merge the abundance table with the selected column
unq_abun = OPUsAb_df_in.merge(unique_table, left_index=True, right_index=True)
# Group by Functional hierarchy 1 and compute the average contribution for each sample
unique_AVGab = unq_abun.groupby('Functional hierarchy 1').mean()
unique_AVGab 

> **ANCOM** (Analysis of Composition of Microbiomes) tests whether the relative abundance of functional metabolic pathways differs significantly across ocean basins, applied to **Endemic (Unique) OPUs**.

> The W-statistic (number of significant pairwise comparisons per feature) is used as the criterion for significance. Results are reported in **Table S6** and **Fig. 2d**.

In [ ]:
metadata_df = pd.read_csv("TableS1_metadata.csv", delimiter=",")
mAncom = metadata_df [['Sample', "Ocean"]]
mAncom = mAncom.set_index("Sample")

In [ ]:
unq = unique_AVGab.T
unq = unq.rename_axis('Sample')
unq = unq.replace(0.000, 1e-6)  # Replace NaN values with a small number as ANCOM does not allow 0 values
unq

In [ ]:
# Run ANCOM
ancom_results = ancom(unq, mAncom['Ocean'])

#Extract significant features
significant_features = ancom_results[0][ancom_results[0]['W'] > 0]  # Higher W values indicate significant differences
print("Significant Metabolic Pathways:")
print(significant_features)

In [ ]:
# Merge with metadata
unq = unq.copy()
unq['Ocean'] = mAncom['Ocean']

# Separate features and group labels
features = unq.drop(columns='Ocean')
groups = unq['Ocean']

# Run ANCOM
ancom_results = ancom(features, groups)

# Define a custom W threshold (conservative)
threshold = 7 #* (features.shape[1] - 1)  # ~60% of comparisons are significant

sig = ancom_results[0][ancom_results[0]['W'] > threshold]
print("Significant Metabolic Pathways (W > {:.0f}):".format(threshold))
print(sig)      


In [ ]:
# Separate features and group labels ===
features = unq.drop(columns='Ocean')
groups = unq['Ocean']


# ANCOM-like con Kruskal-Wallis ===
def ancom_kruskal(features: pd.DataFrame, groups: pd.Series):
    log_features = np.log(features)
    W_stats = {}

    for feature_i in features.columns:
        W = 0
        for feature_j in features.columns:
            if feature_i == feature_j:
                continue

            lr = log_features[feature_i] - log_features[feature_j]
            group_values = [lr[groups == grp] for grp in groups.unique()]
            if all(len(g) > 1 for g in group_values):
                stat, pval = kruskal(*group_values)
                if pval <= 0.01:
                    W += 1
        W_stats[feature_i] = W

    return pd.DataFrame.from_dict(W_stats, orient='index', columns=['W'])

ancom_unq_df = ancom_kruskal(features, groups)
ancom_unq_df

In [ ]:
# Compute CLR difference between groups ===
grouped = features.T
replaced = multiplicative_replacement(grouped)
clr_transformed = pd.DataFrame(clr(replaced), index=grouped.index, columns=grouped.columns).T

clr_mean_by_group = clr_transformed.groupby(groups).mean().T
clr_diff_range = clr_mean_by_group.max(axis=1) - clr_mean_by_group.min(axis=1)
clr_diff_range.name = "CLR_diff"

# Merge + volcano plot ===
merged = ancom_unq_df.merge(clr_diff_range, left_index=True, right_index=True)
threshold = 8
merged["Significant"] = merged["W"] > threshold

# === Step 5: Plot ===
plt.figure(figsize=(11, 6))
sns.scatterplot(data=merged, x="CLR_diff", y="W", hue="Significant",
                palette={True: "red", False: "gray"}, legend=False)

plt.axhline(threshold, ls="--", color="black", label=f"W threshold = {threshold}")

for idx, row in merged.iterrows():
    plt.text(row["CLR_diff"], row["W"], idx, fontsize=9, ha='right')

plt.title("Volcano plot: Kruskal-Wallis ANCOM (W vs CLR difference)")
plt.xlabel("CLR difference (Max - Min across oceans)")
plt.ylabel("W-statistic (Kruskal)")
plt.tight_layout()
plt.savefig("volcano_kruskal_ancom.png", dpi=300)
plt.show()

In [ ]:
pathway_abun = unq_abun.drop(columns=['Functional hierarchy 1']).copy()
pathway_map = unq_abun['Functional hierarchy 1']
grouped = pathway_abun.groupby(pathway_map).sum()  
grouped = grouped.replace(0, 1e-6)  # pseudocount
replaced = multiplicative_replacement(grouped.T)  # samples × pathways
clr_transformed = pd.DataFrame(clr(replaced), index=grouped.columns, columns=grouped.index).T  
clr_transformed


In [ ]:
# === Compute CLR difference between max and min average per ocean ===
clr_mean_by_ocean = clr_transformed.T.groupby(mAncom["Ocean"]).mean().T  # pathways × oceans
clr_diff_range = clr_mean_by_ocean.max(axis=1) - clr_mean_by_ocean.min(axis=1)
clr_diff_range.name = "CLR_diff"

# === Merge with  ANCOM ===
ancom_df = sig
merged = pd.merge(ancom_df, clr_diff_range, left_index=True, right_index=True)

# === Compute significative W ===
threshold = 0.7 * (clr_transformed.shape[0] - 1)
merged["Significant"] = merged["W"] > threshold

# === Volcano plot  ===
rcParams['pdf.fonttype'] = 42  # Ensures text is saved as text, not paths
rcParams['font.family'] = 'Helvetica'
plt.figure(figsize=(11, 6))

sns.scatterplot(data=merged, x="CLR_diff", y="W", hue="Significant",
                palette={True: "red", False: "gray"}, legend=False)

plt.axhline(threshold, ls="--", color="black", label="W threshold")

for idx, row in merged.iterrows():
    plt.text(row["CLR_diff"], row["W"], idx, fontsize=9, ha='right')

plt.title("Volcano plot: W-statistic vs CLR range across oceans")
plt.xlabel("CLR difference (Max - Min across oceans)")
plt.ylabel("W-statistic")
plt.tight_layout()
plt.savefig("CRL_W_endemic.png", dpi=300)
output_pdf = "CRL_W_endemic.pdf"  # Define the PDF file name
plt.savefig(output_pdf, format='pdf', dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Merge metadata with abundance
df = unq.merge(mAncom, left_index=True, right_index=True)
# Merge with metadata to get Ocean info
ab_unq_long = ab_unq_long.merge(mAncom, left_on="Sample", right_index=True)
# Compute average contribution of each Functional hierarchy 1 in each ocean
ocean_avg = ab_unq_long.groupby(["Ocean", "Functional hierarchy 1"])["Abundance"].mean().reset_index()
#ocean_avg.to_csv("Unique_AVG_contribution_of_each_Functional_hierarchy_in_each_ocean_n.csv", index=False)

In [ ]:
# Pivot table for stacking
ocean_pivot = ocean_avg.pivot(index="Ocean", columns="Functional hierarchy 1", values="Abundance")

# Convert to proportions (100% stacked)
ocean_pivot = ocean_pivot.div(ocean_pivot.sum(axis=1), axis=0) * 100  # Normalize to percentages

# Get a colormap (Accent) and create a color list
num_categories = len(ocean_pivot.columns)  # Number of functional groups
colors = cm.tab20(np.linspace(0, 1, num_categories))  # Generate colors from Accent colormap

# Plot
rcParams['pdf.fonttype'] = 42  # Ensures text is saved as text, not paths
rcParams['font.family'] = 'Helvetica'
ocean_pivot.plot(kind="bar", stacked=True, figsize=(10, 6), color=colors)

# Formatting
plt.title("Functional Repertory Across Oceans For Unique OPUs")
plt.ylabel("Percentage Contribution")
plt.xlabel("Ocean")
plt.xticks(rotation=45)
plt.legend(title="Metabolic Pathway", bbox_to_anchor=(1, 1))

# Save the plot as a PDF (before showing the plot)
output_pdf = "Unique_Functional_Repertory_Across_Oceans.pdf"  # Define the PDF file name
plt.savefig(output_pdf, format='pdf', dpi=300, bbox_inches="tight")

# Display the plot
plt.show()

- **Core OPUs:**

- Assess functional diversity variation.  

In [ ]:
# Filter the rows where 'Category' is 'core'
core_table = Un_Cos_KeyKOUnk[Un_Cos_KeyKOUnk['Category'] == 'core']
core_table.shape
#core_table.to_csv("Core_OPUs_KeyKO_Unkn_annotation_50X15.csv", index=False)


In [ ]:
core_table = core_table[['Functional hierarchy 1']]
OPUsAb_df_in = OPUsAb_df.set_index("OPU")
# Merge the abundance table with the selected column
core_ab= OPUsAb_df_in.merge(core_table, left_index=True, right_index=True)
core_ab

In [ ]:
# Group by Functional hierarchy 1 and compute the average contribution for each sample
core_AVGab = core_ab.groupby('Functional hierarchy 1').mean()
core_AVGab

> Same ANCOM approach applied to **Core OPUs**. Identifies conserved metabolic functions that still show quantitative variation across basins. Results reported in **Table S6** and **Fig. 2d**.

In [ ]:
#Test significance using ANCOM.  

cor2 = core_AVGab.T.copy()
cor2.index.name = 'Sample'
cor2 = cor2.replace(0.000, 1e-5)  
cor2['Ocean'] = mAncom.loc[cor2.index, 'Ocean']
cor2


In [ ]:
# Separate features and group labels 
features = cor2.drop(columns='Ocean')
groups = cor2['Ocean']

In [ ]:
#Separate features and group labels 
features = cor2.drop(columns='Ocean')
groups = cor2['Ocean']


# ANCOM-like con Kruskal-Wallis
def ancom_kruskal(features: pd.DataFrame, groups: pd.Series):
    log_features = np.log(features)
    W_stats = {}

    for feature_i in features.columns:
        W = 0
        for feature_j in features.columns:
            if feature_i == feature_j:
                continue

            lr = log_features[feature_i] - log_features[feature_j]
            group_values = [lr[groups == grp] for grp in groups.unique()]
            if all(len(g) > 1 for g in group_values):
                stat, pval = kruskal(*group_values)
                if pval <= 0.001:
                    W += 1
        W_stats[feature_i] = W

    return pd.DataFrame.from_dict(W_stats, orient='index', columns=['W'])

ancom_core_df = ancom_kruskal(features, groups)

ancom_core_df


In [ ]:
# Compute CLR difference between groups 
grouped = features.T
replaced = multiplicative_replacement(grouped)
clr_transformed = pd.DataFrame(clr(replaced), index=grouped.index, columns=grouped.columns).T

clr_mean_by_group = clr_transformed.groupby(groups).mean().T
clr_diff_range = clr_mean_by_group.max(axis=1) - clr_mean_by_group.min(axis=1)
clr_diff_range.name = "CLR_diff"

# Merge + volcano plot ===
merged = ancom_core_df.merge(clr_diff_range, left_index=True, right_index=True)
threshold = 2 #(significativa en al menos el 75% de las comparaciones)
merged["Significant"] = merged["W"] > threshold

# Plot 
rcParams['pdf.fonttype'] = 42  # Ensures text is saved as text, not paths
rcParams['font.family'] = 'Helvetica'
plt.figure(figsize=(11, 6))
sns.scatterplot(data=merged, x="CLR_diff", y="W", hue="Significant",
                palette={True: "red", False: "gray"}, legend=False)

plt.axhline(threshold, ls="--", color="black", label=f"W threshold = {threshold}")

for idx, row in merged.iterrows():
    plt.text(row["CLR_diff"], row["W"], idx, fontsize=9, ha='right')

plt.title("Volcano plot: Kruskal-Wallis ANCOM (W vs CLR difference)")
plt.xlabel("CLR difference (Max - Min across oceans)")
plt.ylabel("W-statistic (Kruskal)")
plt.tight_layout()
plt.savefig("volcano_kruskal_ancom.png", dpi=300)
output_pdf = "volcano_kruskal_ancom_core.pdf"  # Define the PDF file name
plt.savefig(output_pdf, format='pdf', dpi=300, bbox_inches="tight")
plt.show()

In [ ]:

cor = core_AVGab.T
cor = cor.rename_axis('Sample')
cor = cor.replace(0.000, 1e-6)  # Replace NaN values with a small number as ANCOM does not allow 0 values

# Run ANCOM
ancom_results = ancom(cor, mAncom['Ocean'])

# Extract significant features
significant_features = ancom_results[0][ancom_results[0]['W'] > 0]  # Higher W values indicate significant differences

print("Significant Metabolic Pathways:")
print(significant_features)

In [ ]:
#Melt abundance table to long format
core_ab_long = core_ab.melt(id_vars=["Functional hierarchy 1"], var_name="Sample", value_name="Abundance")

# Merge with metadata to get Ocean info
core_ab_long = core_ab_long.merge(mAncom, left_on="Sample", right_index=True)

# Compute average contribution of each Functional hierarchy 1 in each ocean
core_ocean_avg = core_ab_long.groupby(["Ocean", "Functional hierarchy 1"])["Abundance"].mean().reset_index()

#ocean_avg.to_csv("Unique_AVG_contribution_of_each_Functional_hierarchy_in_each_ocean.csv", index=False)

# Pivot table for stacking
ocean_pivot = core_ocean_avg.pivot(index="Ocean", columns="Functional hierarchy 1", values="Abundance")

# Convert to proportions (100% stacked)
ocean_pivot = ocean_pivot.div(ocean_pivot.sum(axis=1), axis=0) * 100  # Normalize to percentages

# Get a colormap (Accent) and create a color list
num_categories = len(ocean_pivot.columns)  # Number of functional groups
colors = cm.Set3(np.linspace(0, 1, num_categories))  # Generate colors from Accent colormap

# Plot
rcParams['pdf.fonttype'] = 42  # Ensures text is saved as text, not paths
rcParams['font.family'] = 'Helvetica'
ocean_pivot.plot(kind="bar", stacked=True, figsize=(10, 6), color=colors)

# Formatting
plt.title("Functional Repertory Across Oceans For Unique OPUs")
plt.ylabel("Percentage Contribution")
plt.xlabel("Ocean")
plt.xticks(rotation=45)
plt.legend(title="Metabolic Pathway", bbox_to_anchor=(1, 1))

# Save the plot as a PDF (before showing the plot)
output_pdf = "Core_Functional_Repertory_Across_Oceans.pdf"  # Define the PDF file name
plt.savefig(output_pdf, format='pdf', dpi=300, bbox_inches="tight")

# Display the plot
plt.show()

- Compute and visualize the contribution of **Known vs. Unknown OPUs** across ocean basins.  

In [ ]:
# Merge OPUs feasture and func. annotation table with ocean binary subsample table on the "OPU" column
mdf = pd.merge(binary_df, OPUsFea_df[['OPU', 'OPU_Type']], on='OPU')
# Melt the DataFrame to get ocean information in a single column
melted_df = mdf.melt(id_vars=['OPU', 'OPU_Type'], var_name='Ocean', value_name='Presence')
# Filter only rows where Presence is 1 (indicating OPU is present in that ocean)
melted_df = melted_df[melted_df['Presence'] == 1]
# Count the number of Known and Unknown OPUs per ocean
opu_counts = melted_df.groupby(['Ocean', 'OPU_Type']).size().unstack(fill_value=0)
# Compute the percentage contribution of Known and Unknown OPUs
opu_percentages = opu_counts.div(opu_counts.sum(axis=1), axis=0) * 100
# Define the approximate locations for each ocean on the map
ocean_locations = {
    'Antarctic': (-80, -85),
    'Arctic': (70, -10),
    'Indian': (-20, 50),
    'Mediterranean': (30, -5),
    'North Atlantic': (25, -50),
    'North Pacific': (30, -150),
    'South Atlantic': (-30, -20),
    'South Pacific': (-30, -120)
}

# Sample data (replace this with opu_percentages data from the previous step)
opu_percentages = {
    'Antarctic': [80, 20],    # [Known, Unknown] percentages
    'Arctic': [82, 18],
    'Indian': [71, 29],
    'Mediterranean': [80, 20],
    'North Atlantic': [74, 26],
    'North Pacific': [73, 27],
    'South Atlantic': [73, 27],
    'South Pacific': [74, 26]
}
# Create the plot with a global map projection
fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={'projection': ccrs.PlateCarree()})
ax.set_global()
ax.coastlines( linewidth=0.2)
ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.2)
ax.add_feature(cfeature.LAND, color='lightgray')
ax.add_feature(cfeature.OCEAN, color='#157fb1')

# Plot each pie chart at the specified ocean location
for ocean, location in ocean_locations.items():
    if ocean in opu_percentages:
        data = opu_percentages[ocean]
        # Create the pie chart as an inset plot with percentages
        ax_inset = ax.inset_axes([location[1] / 360 + 0.5, location[0] / 180 + 0.5, 0.1, 0.1], 
                                 transform=ax.transAxes)
        ax_inset.pie(data, colors=['skyblue', 'salmon'], startangle=90, 
                     autopct='%1.0f', wedgeprops=dict(edgecolor='white'))
        # Label each ocean's pie chart
        ax.text(location[1], location[0], ocean, transform=ccrs.PlateCarree(),
                ha='center', va='center', fontsize=10, fontweight='bold',color='Black')


#plt.title('% Contribution of Known and Unknown OPUs in Each Ocean')
# Save the plot to a file
plt.savefig('Contribution of Known and Unknown OPUs in Each Ocean_map.png', dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()

### **4.3 Functional Diversity and Richness**  

#### **4.3.1 Functional Diversity at OPUs Level (97% Identity) Across Latitude**  

In [ ]:
#Convert Latitude to Absolute Latitude
metadata_df["abs_latitude"] = metadata_df["decimalLatitude"].abs()

In [ ]:
# Compute correlation for Shannon Index
rho_shannon, p_shannon = spearmanr(metadata_df["Shannon_Index"], metadata_df["abs_latitude"])
print(f"Spearman correlation (Shannon Index vs. Latitude): rho={rho_shannon:.3f}, p={p_shannon:.3e}")

# Compute correlation for Chao1 Index
rho_chao, p_chao = spearmanr(metadata_df["Chao1_Index"], metadata_df["abs_latitude"])
print(f"Spearman correlation (Chao1 Index vs. Latitude): rho={rho_chao:.3f}, p={p_chao:.3e}")


In [ ]:
rcParams['pdf.fonttype'] = 42  # Ensures text is saved as text, not paths
rcParams['font.family'] = 'Helvetica'
# Scatterplot for Shannon Index
sns.lmplot(x="abs_latitude", y="Shannon_Index", data=metadata_df, ci=None)
plt.xlabel("Absolute Latitude")
plt.ylabel("Shannon Index")
plt.title(f"Pearson: r={rho_shannon:.3f}, p={p_shannon:.3e}")
plt.savefig("Shannon_vs_Latitude.pdf", format="pdf")  # Save as PDF
plt.show()

# Scatterplot for Chao1 Index
sns.lmplot(x="abs_latitude", y="Chao1_Index", data=metadata_df, ci=None)
plt.xlabel("Absolute Latitude")
plt.ylabel("Chao1 Index")
plt.title(f"Pearson: r={rho_chao:.3f}, p={p_chao:.3e}")
plt.savefig("Chao1_vs_Latitude.pdf", format="pdf")  # Save as PDF
plt.show()

#### **4.3.2 Genome Size Across Latitude**  

In [ ]:
> **Input:** `TableS7_MAGs.txt` – 20,482 marine MAGs with genome size estimates.

> Analyzes the relationship between genome size and absolute latitude using Spearman correlation. Larger genomes in polar waters are consistent with prior literature. Corresponds to **Supplementary Fig. S7**.

In [ ]:
# Group by Latitude and Longitude, then compute the mean genome size for each group
#average_genome_size = mg.groupby(['latitude', 'longitude'])['Estimated_Genome_Size'].mean().reset_index()
average_BGC = mg.groupby(['latitude', 'longitude'])['Biosynthetic Products'].mean().reset_index()

# Rename the resulting columns for clarity
#average_genome_size.columns = ['latitude', 'longitude', 'Average Genome Size']
average_BGC.columns = ['latitude', 'longitude', 'average BGC']

average_BGC["abs_latitude"] = average_BGC["latitude"].abs()
average_BGC

In [ ]:
# Compute correlation for Chao1 Index
#rho_gensize, p_gensize = pearsonr(average_genome_size["Average Genome Size"], average_genome_size["abs_latitude"])
rho_avgBGC, p_avgBGC = spearmanr(average_BGC["average BGC"], average_BGC["abs_latitude"])

print(f"Pearson correlation (average_BGC_size vs. Latitude): rho={rho_avgBGC:.3f}, p={p_avgBGC:.3e}")

In [ ]:
rcParams['pdf.fonttype'] = 42  # Ensures text is saved as text, not paths
rcParams['font.family'] = 'Helvetica'
# Scatterplot for Shannon Index
sns.lmplot(x="abs_latitude", y="average BGC", data=average_BGC, ci=None)
plt.xlabel("Absolute Latitude")
plt.ylabel("Average Genome Size")
plt.title(f"Pearson: r={rho_avgBGC:.3f}, p={p_avgBGC:.3e}")
#plt.savefig("Average Genome Size_vs_Latitude.pdf", format="pdf")  # Save as PDF
plt.show()